# Final analysis: one exact program per network

This notebook reports the completed, frozen degree-five experiment. It reads the new final-analysis artifacts; `release.json` in their directory certifies the complete release after tests and all notebooks finish.

Our object is the **whole ordered one-step output matrix**. The input addresses 0 through 2^N − 1 are implicit, with LSB-first node coordinates. Each network has its own shared symbolic program with N ordered output references. Long-run cycles and basin probabilities are checked and analyzed as separate quantities.

In [ ]:
from pathlib import Path
import hashlib
import sys
import pandas as pd
from IPython.display import display, Markdown, Image
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'doppel-challenge/src').is_dir())
sys.path.insert(0, str(ROOT / 'doppel-challenge/src'))
from doppel_challenge.io import read_json
OUT = ROOT / 'doppel-challenge/results/joint_degree5_final_analysis_v1'
audit = read_json(OUT / 'audit.json')
analysis = read_json(OUT / 'analysis.json')
assert audit['passed'] and not audit['errors']
assert analysis['audit_sha256'] == audit['sha256']
assert hashlib.sha256((OUT / 'network_replay.json').read_bytes()).hexdigest() == audit['network_replay_file_sha256']
networks = pd.read_csv(OUT / 'network_metrics.csv')
effects = pd.read_csv(OUT / 'effect_rows.csv')
bases = pd.read_csv(OUT / 'base_means.csv')
groups = pd.read_csv(OUT / 'group_summary.csv')
assert len(networks) == analysis['n_networks']
assert len(effects) == analysis['n_effects']
assert len(bases) == analysis['n_nonempty_catalogues']
display(pd.DataFrame([{'quantity': k, 'count': analysis[k]} for k in ['n_networks', 'n_bases', 'n_catalogues', 'n_nonempty_catalogues', 'n_effects']]))
print('Fresh replay counts:', audit['stage_counts'])
print('Checked constrained frontiers:', audit['catalogues']['frontiers_verified'])

## 1. What has been checked again?

For every main and pilot network, the final audit rebuilt the symbolic program, checked its bytes against the saved program, replayed independent trajectories, and checked every output bit and the exact long-run probabilities. It also recomputed BDM, the paired effect quantities, and all declared constrained optima.

The independent Wolfram results come from the completed experiment. The audit checks their normal-exit status and agreement with the freshly reconstructed outputs. It does not claim a new full Wolfram run.

A digest identifies exact content; it is not a correctness proof. Canonical record hashes and raw file hashes describe different byte sequences. The previous analysis failure compared these two types. The new audit verifies each against its proper reference and preserves the original failed release for inspection.

In [ ]:
display(pd.DataFrame([{'hash meaning': name, 'digest': audit['inputs'][name]} for name in ['parent_record_sha256', 'parent_file_sha256']]))
assert audit['inputs']['frozen_provenance_matches']
print('Historical recovery attempts verified:', audit['inputs']['recovery_attempts_verified'])

## 2. Raw, our program, and BDM

**Raw** is already binary: N columns × 2^N ordered rows. There is no extra binarization step and no charge for the implicit input table.

**Our program** is one shared executable decision graph per network. Its logical bit length includes the references and decisions needed to reconstruct all outputs. N and the fixed decoder are shared conventions. The serializer's byte padding is not part of this logical length. This is our declared algorithmic description length; it need not be the shortest possible program and can depend on variable order.

**BDM** is a separate estimate computed on the original output matrix, using pybdm 0.1.0 and non-overlapping 4×4 blocks. The N=10 matrix has two zero columns added to make complete blocks: 2,048 padding bits. N=8 and N=12 need no BDM padding. These zeros affect BDM's input, not raw length or our program length.

The table describes unique main-network records. Its medians are descriptive summaries, not independent-replicate uncertainty estimates.

In [ ]:
assert (networks.raw_bits == networks.n.map(lambda n: int(n) * 2**int(n))).all()
networks['program_raw_percent'] = 100 * networks.program_bits / networks.raw_bits
length_table = networks.groupby('n').agg(networks=('network_sha256', 'size'), raw_bits=('raw_bits', 'first'), program_min_bits=('program_bits', 'min'), program_median_bits=('program_bits', 'median'), program_max_bits=('program_bits', 'max'), median_program_raw_percent=('program_raw_percent', 'median'), median_BDM=('bdm', 'median'), BDM_padding_bits=('bdm_padding_bits', 'first'))
display(length_table.round(3))
assert (networks.program_bits < networks.raw_bits).all()
display(Image(filename=str(OUT / 'lengths_and_bdm.png')))

## 3. Give each base network equal weight

Different base networks have different numbers of admissible edge changes. Averaging all perturbation rows gives greater influence to bases with larger catalogues. The protocol instead averages the changes within each base, then gives each base equal weight within its family, size and perturbation kind. Additions and removals are analyzed separately.

The previous overall program/raw mean of about 3.79% describes pooled perturbation rows. It is not the equal-base estimate. The comparison below makes that distinction visible.

One catalogue, modular_n8_s15 / EDGE_REMOVE, has no admissible nonidentity change. It is included among 480 catalogue definitions; it contributes no defined removal-effect mean. That group therefore has 19 contributing bases.

In [ ]:
keys = ['family', 'n', 'kind']
comparison = effects.groupby(keys).program_ratio.mean().rename('pooled_effect_ratio').to_frame().join(bases.groupby(keys).program_ratio.mean().rename('equal_base_ratio'))
display((100 * comparison).round(3).rename(columns=lambda c: c + '_percent'))
display(pd.DataFrame(analysis['empty_catalogues']))
assert len(bases) + len(analysis['empty_catalogues']) == analysis['n_catalogues']

## 4. Program changes, BDM changes, and changes in dynamics

Each row below represents one family/size/kind group. Program change is in bits; BDM change uses BDM's own units. Total variation (TV) measures the change in the long-run distribution, from zero for identical distributions to one for disjoint support. A smaller program does not imply a smaller dynamical change.

The ratio intervals resample base networks 5,000 times within each group. They describe stability across this sample of generated bases. This downstream analysis was developed after inspecting results; these are not preregistered confirmatory tests.

In [ ]:
columns = keys + ['n_bases', 'n_effects', 'mean_delta_program_bits', 'mean_delta_bdm', 'mean_total_variation', 'mean_finite_kl']
display(groups[columns].round(4))
intervals = groups[keys + ['mean_program_ratio', 'program_ratio_lo95', 'program_ratio_hi95']].copy()
for c in ['mean_program_ratio', 'program_ratio_lo95', 'program_ratio_hi95']:
    intervals[c] *= 100
display(intervals.round(3))
display(Image(filename=str(OUT / 'base_associations.png')))
correlations = pd.read_csv(OUT / 'base_correlations.csv')
display(correlations.round(3))

## 5. What these experiments establish

For the frozen main sample, every network's program exactly reconstructs its full one-step output matrix, and every program is shorter than the corresponding raw matrix. The tables show how edge additions and removals change program length, BDM and long-run dynamics. The scatter plots pool groups for display; the correlation table keeps family, size and perturbation kind separate and uses base means.

The degree-five construction limits the size of local Boolean functions, so this success does not establish unrestricted scaling. Exhaustive means every input state and every declared admissible one-edge perturbation for the selected bases. The sampled N=100 benchmark is outside this exact experimental claim.

The original notebook 02 remains a pre-analysis explanation. The final tables, weighting and validation evidence in this notebook and the final-analysis directory govern the completed downstream analysis. The experiment does not establish universal Kolmogorov complexity, identify network mechanisms from output distributions, or validate a detector.